# 01 Jobs Basics

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Turn a notebook into a scheduled, production job with <b>Lakeflow Jobs</b> (formerly Databricks Workflows): understand jobs, tasks, compute, triggers, parameters and runs, write task code that is safe to re-run, create and run a job from code with the Databricks SDK, and see the same job as configuration you can keep in Git.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Notebooks run by hand aren't pipelines. Jobs run your code on a schedule or trigger, with the right compute, parameters, retries and alerts. Lakeflow Jobs is 16% of the Data Engineer Associate exam, and "how do you schedule and operate this pipeline?" follows every technical interview question about building one.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Anatomy of a job</b><br>
<code>`</code><br>Job  (name, owner / run-as, parameters, triggers, notifications, max concurrent runs, tags)<br>├── Task "ingest"      type: notebook | Python script | Python wheel | SQL | pipeline | dbt | JAR | run job | if/else | for each<br>│     compute: serverless (default in Free Edition) | job cluster | (all-purpose: avoid for production)<br>├── Task "transform"   depends on "ingest"<br>└── Task "publish"     depends on "transform"<br>Run = one execution of the job; task run = one execution of a task (with attempts if retried)<br><code>`</code><br><br>
| Trigger | Starts a run when... |<br>|---|---|<br>| Manual / API | Someone clicks <b>Run now</b> or calls the API |<br>| Scheduled | A cron expression fires (Quartz syntax, with a time zone) |<br>| File arrival | New files land in a volume or external location |<br>| Table update | Source tables are updated |<br>| Continuous | The previous run ends (always running) |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team ran a "daily load" notebook by hand every morning. When the engineer was on leave, nobody ran it, and when it was run twice by mistake, sales doubled in the dashboard. They turned it into a job: scheduled at 06:00 UTC, with a <code>run_date</code> parameter, on serverless compute, with failure emails, and the notebook was changed to overwrite only the day it loads. Missed days are now backfilled by running the job with an older <code>run_date</code>, and double runs change nothing.
</div>

## Syntax

```python
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()                                    # authenticates automatically inside Databricks
job = w.jobs.create(
    name="daily_orders",
    parameters=[jobs.JobParameterDefinition(name="run_date", default="2024-11-01")],
    tasks=[jobs.Task(task_key="ingest", notebook_task=jobs.NotebookTask(notebook_path="/Users/me/ingest"))],
    schedule=jobs.CronSchedule(quartz_cron_expression="0 0 6 * * ?", timezone_id="UTC"),
)
w.jobs.run_now(job_id=job.job_id, job_parameters={"run_date": "2024-11-02"}).result()
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema <code>workflows</code> for this folder, and (on Databricks) an SDK client plus a helper that writes notebooks into your workspace folder <code>databricks-learning-workflows</code>.<br><br>
<b>Why it matters</b><br>Jobs run notebooks that live in the workspace (or in Git). The helper creates small task notebooks for the demos, so you don't have to create them by hand.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your user name and the workspace folder used for the demo notebooks.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.workflows"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

import base64, json, time
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs, workspace

w = WorkspaceClient()
me = w.current_user.me().user_name
FOLDER = f"/Users/{me}/databricks-learning-workflows"
w.workspace.mkdirs(FOLDER)

def put_notebook(name, source, language=workspace.Language.PYTHON):
    path = f"{FOLDER}/{name}"
    w.workspace.import_(path=path, format=workspace.ImportFormat.SOURCE, language=language,
                        content=base64.b64encode(source.encode()).decode(), overwrite=True)
    return path

print("user:", me, "| folder:", FOLDER)

## 1. Write task code that is safe to re-run

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines the logic of the daily load as a function: it loads one day of orders and <b>replaces only that day</b> in the target table. Runs it twice for the same date and once for another date.<br><br>
<b>Why it matters</b><br>Jobs get retried, re-run and backfilled. Task code must be <b>idempotent</b>: running it twice for the same input gives the same result. <code>replaceWhere</code> (or <code>MERGE</code>) achieves that. A plain <code>append</code> would duplicate the day.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After two runs for 2024-11-01, the table still has 100 rows for that day. After loading 2024-11-02, 200 rows in total.
</div>

In [0]:
from pyspark.sql import functions as F

TARGET = f"{DB}.daily_orders"
spark.sql(f"CREATE TABLE IF NOT EXISTS {TARGET} (order_date DATE, order_id BIGINT, amount DOUBLE)")

def load_day(run_date):
    day = (spark.range(100)
           .select(F.lit(run_date).cast("date").alias("order_date"), F.col("id").alias("order_id"),
                   (F.col("id") % 7 * 10.0).alias("amount")))
    day.write.mode("overwrite").option("replaceWhere", f"order_date = '{run_date}'").saveAsTable(TARGET)
    return {"run_date": run_date, "rows": day.count()}

print(load_day("2024-11-01"))
print(load_day("2024-11-01"))   # re-run: same result, no duplicates
print(load_day("2024-11-02"))
spark.table(TARGET).groupBy("order_date").count().orderBy("order_date").show()

## 2. Put the task in a notebook

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a notebook <code>ingest_daily_orders</code> in your workspace folder. It reads the <code>run_date</code> parameter with a widget, runs the same load, and returns a small JSON result with <code>dbutils.notebook.exit</code>.<br><br>
<b>Why it matters</b><br>Job parameters reach notebook tasks as <b>widgets</b>, so the same notebook works interactively (with the widget's default) and in a job (with the job's value). <code>dbutils.notebook.exit</code> returns a value that shows up in the run output.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The path of the created notebook. Open it from the workspace browser to see the code.
</div>

In [0]:
INGEST_SRC = f'''# Databricks notebook source
import json
from pyspark.sql import functions as F

dbutils.widgets.text("run_date", "2024-11-01")
run_date = dbutils.widgets.get("run_date")
TARGET = "{TARGET}"

spark.sql(f"CREATE TABLE IF NOT EXISTS {{TARGET}} (order_date DATE, order_id BIGINT, amount DOUBLE)")
day = (spark.range(100)
       .select(F.lit(run_date).cast("date").alias("order_date"), F.col("id").alias("order_id"),
               (F.col("id") % 7 * 10.0).alias("amount")))
day.write.mode("overwrite").option("replaceWhere", f"order_date = '{{run_date}}'").saveAsTable(TARGET)
dbutils.notebook.exit(json.dumps({{"run_date": run_date, "rows": day.count()}}))
'''
ingest_path = put_notebook("ingest_daily_orders", INGEST_SRC)
print(ingest_path)

## 3. Create the job

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a job with one notebook task on serverless compute (no cluster settings means serverless), a <code>run_date</code> job parameter, a tag, and at most one concurrent run.<br><br>
<b>Why it matters</b><br>Everything you can click in <b>Jobs & Pipelines</b> can be done from code, which makes jobs reproducible. Limiting concurrent runs prevents two runs of a daily load from overlapping.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Free Edition supports jobs on serverless compute, with limits on concurrent runs. If job creation fails, the cell prints the error. Create the same job in the UI instead: <b>Jobs & Pipelines</b> → <b>Create</b> → <b>Job</b>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A job ID and a link to the job.
</div>

In [0]:
job = w.jobs.create(
    name="dbl_01_daily_orders",
    parameters=[jobs.JobParameterDefinition(name="run_date", default="2024-11-01")],
    tasks=[jobs.Task(task_key="ingest", notebook_task=jobs.NotebookTask(notebook_path=ingest_path))],
    max_concurrent_runs=1,
    tags={"course": "databricks-learning", "lesson": "07_01"},
)
JOB_ID = job.job_id
print("job id:", JOB_ID, "| open:", f"{w.config.host}/jobs/{JOB_ID}")

## 4. Run it and read the result

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs the job for 2024-11-03, waits for it to finish, and prints the run's state and the notebook's exit value.<br><br>
<b>Why it matters</b><br><code>run_now</code> returns a waiter: <code>.result()</code> blocks until the run ends. The run page link shows the same information in the UI, including the notebook as it ran.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>SUCCESS</code> (serverless start-up can take a minute), the exit value <code>{"run_date": "2024-11-03", "rows": 100}</code>, and a third day in the table.
</div>

In [0]:
run = w.jobs.run_now(job_id=JOB_ID, job_parameters={"run_date": "2024-11-03"}).result(timeout=timedelta(minutes=20))
print("result:", run.state.result_state.value if run.state.result_state else run.state.life_cycle_state.value)
print("run page:", run.run_page_url)
task_run = run.tasks[0]
print("notebook exit value:", w.jobs.get_run_output(run_id=task_run.run_id).notebook_output.result)
spark.table(TARGET).groupBy("order_date").count().orderBy("order_date").show()

## 5. Add a schedule (paused)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a daily 06:00 UTC schedule to the job, created <b>paused</b> so it doesn't actually run every day.<br><br>
<b>Why it matters</b><br>Schedules use Quartz cron syntax: seconds, minutes, hours, day of month, month, day of week. Always set the time zone explicitly. Pausing a schedule is how you stop a job temporarily without losing its configuration.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The job's schedule with its cron expression and <code>PAUSED</code> status.
</div>

| Quartz cron | Meaning |
|---|---|
| `0 0 6 * * ?` | Every day at 06:00 |
| `0 30 * * * ?` | Every hour at minute 30 |
| `0 0 7 ? * MON-FRI` | Weekdays at 07:00 |
| `0 0 2 1 * ?` | First day of every month at 02:00 |

In [0]:
w.jobs.update(job_id=JOB_ID, new_settings=jobs.JobSettings(
    schedule=jobs.CronSchedule(quartz_cron_expression="0 0 6 * * ?", timezone_id="UTC",
                               pause_status=jobs.PauseStatus.PAUSED)))
s = w.jobs.get(job_id=JOB_ID).settings.schedule
print(s.quartz_cron_expression, s.timezone_id, s.pause_status.value)

## 6. The job as configuration

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the job's settings as JSON from the API, and the equivalent <b>Databricks Asset Bundles</b> YAML you'd keep in Git.<br><br>
<b>Why it matters</b><br>In production, jobs are defined as code and deployed with bundles (lesson 07 and <code>09_azure_databricks</code>). The UI is great for exploring. Configuration in Git is what you review, version and promote from dev to prod.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The job settings JSON (on Databricks), and the YAML.
</div>

In [0]:
print(json.dumps(w.jobs.get(job_id=JOB_ID).settings.as_dict(), indent=2)[:1500])

In [0]:
print("""
# resources/daily_orders.job.yml
resources:
  jobs:
    daily_orders:
      name: daily_orders
      max_concurrent_runs: 1
      parameters:
        - name: run_date
          default: "2024-11-01"
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: UTC
        pause_status: PAUSED
      tasks:
        - task_key: ingest
          notebook_task:
            notebook_path: ../src/ingest_daily_orders.py
      tags:
        course: databricks-learning
""")

## 7. Clean up

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deletes the demo job. Set <code>KEEP_JOB = True</code> first if you want to explore it in the UI.<br><br>
<b>Why it matters</b><br>Unused jobs with schedules keep running and costing money. Delete what you don't need.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>job deleted</code> (or <code>job kept</code>).
</div>

In [0]:
KEEP_JOB = False
if KEEP_JOB:
    print("job kept:", f"{w.config.host}/jobs/{JOB_ID}")
else:
    w.jobs.delete(job_id=JOB_ID)
    print("job deleted")

## Under the hood

```
trigger (schedule / file arrival / API) ─▶ Jobs service creates a run (checks max_concurrent_runs, queue)
   └─ for each task in dependency order:
        compute: serverless (starts in seconds) or a new job cluster
        notebook gets job parameters as widgets ─▶ runs as the job's "run as" identity
        result: SUCCESS / FAILED / TIMEDOUT / CANCELED  (+ exit value, logs, Spark UI)
run history kept per job ─▶ UI, API (jobs.list_runs), system.lakeflow tables
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> the notebook's writes run with the permissions of the job's **run as** user or service principal, not of whoever clicked Run.

In [0]:
spark.sql(f"DESCRIBE HISTORY {TARGET}").select("version", "operation", "operationParameters").show(5, truncate=60)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: production jobs on all-purpose clusters</b><br>
Higher cost, shared with interactive users. Use serverless or job compute.<br><br>
<b>⛔ Problem: duplicate data after a retry or re-run</b><br>The task appends. Make it idempotent with <code>replaceWhere</code>, <code>MERGE</code> or <code>INSERT OVERWRITE</code> for the processed slice.<br><br>
<b>⛔ Problem: job uses "today" instead of a parameter</b><br>Backfills become impossible. Pass <code>run_date</code> as a parameter, with a default or <code>{{job.start_time.iso_date}}</code>.<br><br>
<b>⛔ Problem: job runs as a person who has left</b><br>Set <b>run as</b> to a service principal for production jobs.<br><br>
<b>⛔ Problem: two runs overlap and corrupt state</b><br>Set <code>max_concurrent_runs = 1</code> (and enable queueing).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What are the main parts of a Databricks job?</b><br>
Tasks (notebooks, scripts, wheels, SQL, pipelines, dbt, and control-flow tasks) with dependencies, compute for each task (serverless or job clusters), triggers (schedule, file arrival, table update, continuous, manual), job parameters, notifications, retries and timeouts, permissions, and the run-as identity.<br><br>

<b>🎤 2. Why use job compute or serverless instead of an all-purpose cluster for jobs?</b><br>
They're cheaper per DBU, isolated per run, and exist only while the job runs. All-purpose clusters are meant for interactive work and keep running (and costing) between jobs.<br><br>

<b>🎤 3. How do you make a daily job safe to re-run and backfill?</b><br>
Parameterize the date (<code>run_date</code>), and make the write idempotent for that slice: overwrite with <code>replaceWhere</code>, <code>MERGE</code> on keys, or <code>INSERT OVERWRITE</code> the partition. Then re-runs and backfills give the same result.<br><br>

<b>🎤 4. How do job parameters reach a notebook task?</b><br>
As widgets: the notebook reads them with <code>dbutils.widgets.get("name")</code>. Defaults come from the job parameter definition, and values can be overridden per run.<br><br>

<b>🎤 5. How can you define jobs as code?</b><br>
With Databricks Asset Bundles (YAML resources deployed with the Databricks CLI), the Jobs REST API or SDK, or Terraform. Bundles are the standard way to version jobs in Git and promote them between environments.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A daily job must start at 05:30 UTC every day. Which schedule configuration is correct?</b><br>
A. Quartz cron <code>0 30 5 * * ?</code> with time zone UTC<br>
B. Quartz cron <code>30 5 * * *</code> with no time zone<br>
C. A file arrival trigger<br>
D. A continuous trigger<br>
<details><summary><b>Show answer</b></summary><b>A.</b></details><br><br>

<b>Q2. Which compute is recommended for a scheduled production notebook task?</b><br>
A. A shared all-purpose cluster that runs 24/7<br>
B. Serverless compute for jobs, or a job cluster<br>
C. A SQL warehouse<br>
D. The driver of another job<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. How does a notebook task read a job parameter named <code>run_date</code>?</b><br>
A. <code>spark.conf.get("run_date")</code><br>
B. <code>dbutils.widgets.get("run_date")</code><br>
C. <code>os.environ["run_date"]</code><br>
D. <code>dbutils.jobs.get("run_date")</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Change <code>load_day</code> so it also writes a <code>loaded_at</code> timestamp, and confirm that re-running a day still leaves exactly 100 rows for it</li><li>Write the Quartz cron expressions for: every 15 minutes, and Saturdays at 01:00</li><li>(Databricks) Create the job again, run it for three different dates in a loop (a mini backfill), then delete it</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Idempotent load with a load timestamp
spark.sql(f"CREATE OR REPLACE TABLE {DB}.daily_orders_v2 (order_date DATE, order_id BIGINT, amount DOUBLE, loaded_at TIMESTAMP)")

def load_day_v2(run_date):
    day = (spark.range(100)
           .select(F.lit(run_date).cast("date").alias("order_date"), F.col("id").alias("order_id"),
                   (F.col("id") % 7 * 10.0).alias("amount"), F.current_timestamp().alias("loaded_at")))
    day.write.mode("overwrite").option("replaceWhere", f"order_date = '{run_date}'").saveAsTable(f"{DB}.daily_orders_v2")

load_day_v2("2024-11-01"); load_day_v2("2024-11-01")
print("rows for 2024-11-01:", spark.table(f"{DB}.daily_orders_v2").filter("order_date = '2024-11-01'").count())

# 2. Quartz cron
print("every 15 minutes:      0 0/15 * * * ?")
print("Saturdays at 01:00:    0 0 1 ? * SAT")

In [0]:
# 3. Mini backfill
job = w.jobs.create(name="dbl_01_backfill",
                    parameters=[jobs.JobParameterDefinition(name="run_date", default="2024-11-01")],
                    tasks=[jobs.Task(task_key="ingest", notebook_task=jobs.NotebookTask(notebook_path=ingest_path))])
for d in ["2024-11-04", "2024-11-05", "2024-11-06"]:
    r = w.jobs.run_now(job_id=job.job_id, job_parameters={"run_date": d}).result(timeout=timedelta(minutes=20))
    print(d, r.state.result_state.value if r.state.result_state else r.state.life_cycle_state.value)
w.jobs.delete(job_id=job.job_id)
spark.table(TARGET).groupBy("order_date").count().orderBy("order_date").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>A job = tasks + compute + triggers + parameters + notifications, runs as an identity</li><li>Use serverless or job compute for production. Avoid all-purpose clusters</li><li>Make tasks idempotent and parameterized (<code>run_date</code>) so retries, re-runs and backfills are safe</li><li>Job parameters arrive as widgets. Return values with <code>dbutils.notebook.exit</code></li><li>Schedules use Quartz cron plus a time zone. Pause instead of deleting</li><li>Define jobs as code (bundles, SDK, API) and keep them in Git</li></ul>
</div>

| Task | SDK |
|---|---|
| Client | `w = WorkspaceClient()` |
| Create | `w.jobs.create(name=..., tasks=[jobs.Task(...)])` |
| Run and wait | `w.jobs.run_now(job_id=..., job_parameters={...}).result()` |
| Task output | `w.jobs.get_run_output(run_id=task_run_id)` |
| Schedule | `jobs.CronSchedule(quartz_cron_expression=..., timezone_id="UTC")` |
| Delete | `w.jobs.delete(job_id=...)` |

## Git commit

```
git add 07_workflows/01_jobs_basics.ipynb
git commit -m "add 07_01 jobs basics notebook"
```